# Experimentos da Árvore B em memória secundária

Este notebook **envelopa o binário C++** (`bench_<ordem>`) e mede, a partir dos
contadores do próprio C++, o custo das operações da B-Tree:

- **leituras / escritas em disco** (contadores do `BTreeManager`)
- **tempo** (medido em C++ com `<chrono>`, só em volta do laço da operação)

Três experimentos, cada um salvo em sua própria subpasta de `experimentos/`:

1. **EXP1 — Operações × Ordem**, *uma curva por tamanho de cache*. Linha vermelha
   tracejada marca `ORDER = CACHE`.
2. **EXP2 — Operações × Tamanho** do dataset (ordem e cache fixos).
3. **EXP3 — Operações × Cache**, *uma curva por ordem* (espelho do EXP1). Linha
   vermelha tracejada marca `ORDER = CACHE`.

> **Tempo de execução.** A matriz completa leva ~20-30 min (EXP2 inclui 1M, e
> ordens baixas com cache pequeno reler muito disco). Rode o **smoke test** antes.

## 1. Imports e constantes

In [1]:
import subprocess, os, sys
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # sem display; salva PNGs direto
import matplotlib.pyplot as plt

ROOT = Path.cwd()
DATASETS     = ROOT / "datasets"
ARVORES      = ROOT / "arvores"
EXPERIMENTOS = ROOT / "experimentos"
EXP1_DIR = EXPERIMENTOS / "exp1"
EXP2_DIR = EXPERIMENTOS / "exp2"
EXP3_DIR = EXPERIMENTOS / "exp3"
for d in (DATASETS, ARVORES, EXP1_DIR, EXP2_DIR, EXP3_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---- EXP1: varia ORDER; uma curva por cache. Linha em ORDER = CACHE. ----
EXP1_SIZE   = 100_000
EXP1_CACHES = [4, 8, 16]                                  # cada cache vira uma curva
ORDERS      = [3, 4, 5, 6, 8, 10, 12, 16, 20, 24, 32]     

# ---- EXP2: varia tamanho; ordem e cache fixos. ----
EXP2_ORDER, EXP2_CACHE = 8, 3
SIZES = [100_000, 250_000, 500_000, 1_000_000]

# ---- EXP3: varia CACHE; uma curva por ordem. Linha em ORDER = CACHE. ----
EXP3_SIZE   = 100_000
EXP3_ORDERS = [4, 8, 16]                                  # cada ordem vira uma curva
CACHES      = [2, 3, 4, 5, 6, 7, 8, 10, 12, 16, 20, 24, 32]  

SEARCH_N = 10_000   # nº de buscas por medicao
SEED     = 42

COLS = ["order", "size", "cache", "phase", "n", "time_s", "reads", "writes", "found"]
print("Diretorios prontos.")

Diretorios prontos.


## 2. Helpers (build, datasets, runs, parsing, HTML)

In [2]:
def sh(cmd):
    "Roda comando (lista de args) e devolve stdout; levanta em erro."
    r = subprocess.run([str(c) for c in cmd], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("Falhou: " + " ".join(map(str, cmd)) + "\nstderr:\n" + r.stderr)
    return r.stdout

def ensure_bench(order):
    "Compila bin/bench_<order> se necessario (make idempotente)."
    sh(["make", f"bin/bench_{order}"])
    return f"./bin/bench_{order}"

def ensure_dataset(size):
    "Gera datasets/n<size>.csv se faltar."
    path = DATASETS / f"n{size}.csv"
    if not path.exists():
        sh([sys.executable, "generate_dataset.py", "-n", size, "-o", path, "-s", SEED])
    return str(path)

def tree_path(order, size):
    return str(ARVORES / f"o{order}_n{size}.dat")

def parse_result(stdout):
    "Extrai a linha 'RESULT,...' e devolve dict tipado."
    line = next(l for l in stdout.splitlines() if l.startswith("RESULT"))
    d = dict(tok.split("=") for tok in line.split(",")[1:])
    return {"order": int(d["order"]), "cache": int(d["cache"]), "phase": d["phase"],
            "n": int(d["n"]), "time_s": float(d["time_s"]), "reads": int(d["reads"]),
            "writes": int(d["writes"]), "found": int(d["found"])}

def run_insert(order, size, cache, tree):
    "Mede insercao com build FRESCO da arvore em <tree>."
    for ext in ("", ".meta"):
        p = Path(tree + ext)
        if p.exists(): p.unlink()
    ds = ensure_dataset(size)
    bench = ensure_bench(order)
    out = sh([bench, "--phase", "insert", "--tree", tree, "--dataset", ds, "--cache", cache])
    row = parse_result(out); row["size"] = size
    return row

def run_search(order, size, cache, tree, n=SEARCH_N):
    "Mede busca reusando arvore ja construida em <tree>."
    bench = ensure_bench(order)
    out = sh([bench, "--phase", "search", "--tree", tree, "--cache", cache,
              "--n", n, "--n-keys", size, "--seed", SEED])
    row = parse_result(out); row["size"] = size
    return row

def ensure_tree(order, size, build_cache=64):
    "Constroi a arvore canonica (NAO-medicao) se faltar. Estrutura = f(order, size)."
    tree = tree_path(order, size)
    if not Path(tree).exists():
        ds = ensure_dataset(size)
        bench = ensure_bench(order)
        sh([bench, "--phase", "insert", "--tree", tree, "--dataset", ds, "--cache", build_cache])
    return tree

def snapshot_html(order, size, outdir, max_depth=3):
    "Gera HTML da arvore (SEMPRE com max-depth pequeno) em <outdir>."
    tree = ensure_tree(order, size)
    out = Path(outdir) / f"tree_o{order}_n{size}.html"
    sh([sys.executable, "generate_images.py", tree, "--order", order,
        "--max-depth", max_depth, "-o", out])
    return str(out)

## 3. Smoke test (rodar PRIMEIRO)

Parâmetros minúsculos só para validar o pipeline ponta-a-ponta antes da matriz.

In [3]:
def smoke():
    orders, sizes, caches, nq = [3, 8], [1000], [2, 10], 200
    for o in orders:
        for s in sizes:
            ensure_dataset(s)
            for c in caches:
                tree = tree_path(o, s)
                ins = run_insert(o, s, c, tree)
                assert ins["writes"] > 0, "insert deveria gravar em disco"
                assert ins["time_s"] > 0, "tempo de insert deveria ser > 0"
                sch = run_search(o, s, c, tree, n=nq)
                assert 0 < sch["found"] < nq, f"found fora do esperado: {sch['found']}"
            assert Path(tree_path(o, s)).exists(), "arvore canonica nao persistida"
    html = snapshot_html(3, 1000, EXPERIMENTOS)
    assert os.path.getsize(html) < 1_000_000, "HTML grande demais (faltou max-depth?)"
    os.remove(html)  # nao deixa lixo do smoke
    print("SMOKE OK")

smoke()

SMOKE OK


## 4. EXP1 — Operações × Ordem (uma curva por cache)

Tamanho fixo (`EXP1_SIZE`); para cada cache em `EXP1_CACHES` varremos `ORDERS`.
Cada `(ordem, cache)` faz build fresco (insert medido) e reusa a árvore na busca.

In [4]:
rows = []
for c in EXP1_CACHES:
    for o in ORDERS:
        tree = tree_path(o, EXP1_SIZE)
        rows.append(run_insert(o, EXP1_SIZE, c, tree))
        rows.append(run_search(o, EXP1_SIZE, c, tree))
df_exp1 = pd.DataFrame(rows)[COLS]
df_exp1.to_csv(EXP1_DIR / "exp1.csv", index=False)
df_exp1

,order,size,cache,phase,n,time_s,reads,writes,found
0,3,100000,4,insert,100000,19.782800,1568754,349941,0
1,3,100000,4,search,10000,1.342240,164945,0,5000
2,4,100000,4,insert,100000,18.831800,1468752,349873,0
3,4,100000,4,search,10000,1.277040,154944,0,5000
4,5,100000,4,insert,100000,12.480600,966738,216471,0
...,...,...,...,...,...,...,...,...,...
61,20,100000,16,search,10000,0.150744,17182,0,5000
62,24,100000,16,insert,100000,0.195766,0,9089,0
63,24,100000,16,search,10000,0.140467,15686,0,5000
64,32,100000,16,insert,100000,0.151805,0,6665,0


## 5. EXP2 — Operações × Tamanho

Ordem (`EXP2_ORDER`) e cache (`EXP2_CACHE`) fixos; varia o nº de registros.
**Inclui 1M — pode demorar.**

### Por que `exp2_writes_search.png` é uma reta em 0?
A busca (`mSearch`) é **somente leitura**: percorre nós comparando chaves, mas
nunca os modifica. Como nenhum nó fica *dirty*, `saveNode` jamais é chamado na
busca → **`writes = 0` sempre**, independentemente do tamanho do dataset. Não é
bug; é definição. Isso vale para os três experimentos — por isso os gráficos de
*writes* na fase *search* são **omitidos** (seriam sempre zero).

In [5]:
rows = []
for s in SIZES:
    tree = tree_path(EXP2_ORDER, s)
    rows.append(run_insert(EXP2_ORDER, s, EXP2_CACHE, tree))
    rows.append(run_search(EXP2_ORDER, s, EXP2_CACHE, tree))
df_exp2 = pd.DataFrame(rows)[COLS]
df_exp2.to_csv(EXP2_DIR / "exp2.csv", index=False)
df_exp2

,order,size,cache,phase,n,time_s,reads,writes,found
0,8,100000,3,insert,100000,10.735000,755975,174805,0
1,8,100000,3,search,10000,0.701050,78312,0,5000
2,8,250000,3,insert,250000,27.319600,2074896,437305,0
3,8,250000,3,search,10000,0.735987,88376,0,5000
4,8,500000,3,insert,500000,52.424300,4324896,874801,0
5,8,500000,3,search,10000,0.764509,88335,0,5000
6,8,1000000,3,insert,1000000,111.094000,9300600,1749799,0
7,8,1000000,3,search,10000,0.859022,98352,0,5000


## 6. EXP3 — Operações × Cache (uma curva por ordem)

Espelho do EXP1: tamanho fixo (`EXP3_SIZE`); para cada ordem em `EXP3_ORDERS`
varremos `CACHES`. O insert é remedido por cache numa árvore descartável
(`_tmp_build.dat`); a busca reusa a árvore canônica daquela ordem (idêntica para
todo cache).

In [6]:
tmp = str(ARVORES / "_tmp_build.dat")
rows = []
for o in EXP3_ORDERS:
    canon = ensure_tree(o, EXP3_SIZE)            # canonica p/ a busca (build unico)
    for c in CACHES:
        rows.append(run_insert(o, EXP3_SIZE, c, tmp))    # insert medido por cache
        rows.append(run_search(o, EXP3_SIZE, c, canon))  # search reusa canonica
for ext in ("", ".meta"):
    p = Path(tmp + ext)
    if p.exists(): p.unlink()
df_exp3 = pd.DataFrame(rows)[COLS]
df_exp3.to_csv(EXP3_DIR / "exp3.csv", index=False)
df_exp3

,order,size,cache,phase,n,time_s,reads,writes,found
0,4,100000,2,insert,100000,20.401100,1468814,349913,0
1,4,100000,2,search,10000,1.412540,154945,0,5000
2,4,100000,3,insert,100000,19.937500,1468795,349900,0
3,4,100000,3,search,10000,1.316480,154945,0,5000
4,4,100000,4,insert,100000,19.326700,1468752,349873,0
...,...,...,...,...,...,...,...,...,...
73,16,100000,20,search,10000,0.171993,19263,0,5000
74,16,100000,24,insert,100000,0.308010,0,14285,0
75,16,100000,24,search,10000,0.167193,18750,0,5000
76,16,100000,32,insert,100000,0.303428,0,14285,0


## 7. Gráficos

- `plot_series`: várias curvas (uma por valor de `series_col`) + **linha vermelha
  tracejada em `ORDER = CACHE`** (uma por valor da série). Usado em EXP1 e EXP3.
- `plot_simple`: curva única (EXP2).
- Em ambos, o gráfico *writes × search* é pulado (writes é sempre 0 na busca).

In [7]:
PHASES = ("insert", "search")
METRICS = ("time_s", "reads", "writes")

def _skip(metric, phase):
    return metric == "writes" and phase == "search"  # busca nunca grava

def plot_series(df, x, series_col, outdir, prefix, logx=False):
    for metric in METRICS:
        for phase in PHASES:
            if _skip(metric, phase):
                continue
            sub = df[df["phase"] == phase]
            if sub.empty:
                continue
            vals = sorted(sub[series_col].unique())
            fig, ax = plt.subplots(figsize=(7, 4.5))
            for v in vals:
                s = sub[sub[series_col] == v].sort_values(x)
                ax.plot(s[x], s[metric], marker="o", label=f"{series_col}={v}")
            # linha vertical vermelha tracejada onde ORDER == CACHE
            for v in vals:
                ax.axvline(v, color="red", ls="--", lw=1, alpha=0.6)
            ax.plot([], [], color="red", ls="--", lw=1, label="ORDER = CACHE")
            if logx:
                ax.set_xscale("log")
            ax.set_xlabel(x)
            ax.set_ylabel(metric)
            ax.set_title(f"{prefix}: {metric} ({phase})")
            ax.grid(True, alpha=0.3)
            ax.legend(fontsize=8)
            fig.tight_layout()
            fig.savefig(outdir / f"{prefix}_{metric}_{phase}.png", dpi=120)
            plt.close(fig)

def plot_simple(df, x, outdir, prefix, logx=False):
    for metric in METRICS:
        for phase in PHASES:
            if _skip(metric, phase):
                continue
            sub = df[df["phase"] == phase].sort_values(x)
            if sub.empty:
                continue
            fig, ax = plt.subplots(figsize=(6, 4))
            ax.plot(sub[x], sub[metric], marker="o")
            if logx:
                ax.set_xscale("log")
            ax.set_xlabel(x)
            ax.set_ylabel(metric)
            ax.set_title(f"{prefix}: {metric} ({phase})")
            ax.grid(True, alpha=0.3)
            fig.tight_layout()
            fig.savefig(outdir / f"{prefix}_{metric}_{phase}.png", dpi=120)
            plt.close(fig)

plot_series(df_exp1, "order", "cache", EXP1_DIR, "exp1", logx=True)
plot_simple(df_exp2, "size",          EXP2_DIR, "exp2", logx=True)
plot_series(df_exp3, "cache", "order", EXP3_DIR, "exp3", logx=True)
print("PNGs salvos em experimentos/exp1, exp2, exp3")

PNGs salvos em experimentos/exp1, exp2, exp3


## 8. Snapshots HTML das árvores (sempre com max-depth pequeno)

In [8]:
# EXP1 / EXP3: duas ordens ilustrativas no tamanho do experimento
for o in (4, 16):
    print(snapshot_html(o, EXP1_SIZE, EXP1_DIR))
    print(snapshot_html(o, EXP3_SIZE, EXP3_DIR))
# EXP2: dois tamanhos na ordem do experimento (max-depth garante HTML pequeno)
for s in (100_000, 1_000_000):
    print(snapshot_html(EXP2_ORDER, s, EXP2_DIR))

/Users/antonio/Documents/Estudos/projetos_de_estudo/algoritmos_estruturas_dados/experimentos/exp1/tree_o4_n100000.html
/Users/antonio/Documents/Estudos/projetos_de_estudo/algoritmos_estruturas_dados/experimentos/exp3/tree_o4_n100000.html
/Users/antonio/Documents/Estudos/projetos_de_estudo/algoritmos_estruturas_dados/experimentos/exp1/tree_o16_n100000.html
/Users/antonio/Documents/Estudos/projetos_de_estudo/algoritmos_estruturas_dados/experimentos/exp3/tree_o16_n100000.html
/Users/antonio/Documents/Estudos/projetos_de_estudo/algoritmos_estruturas_dados/experimentos/exp2/tree_o8_n100000.html
/Users/antonio/Documents/Estudos/projetos_de_estudo/algoritmos_estruturas_dados/experimentos/exp2/tree_o8_n1000000.html


In [15]:
mask = (df_exp1["order"]<=16) & (df_exp1["cache"]==16) & (df_exp1["phase"]=="insert")
directory_output = EXPERIMENTOS / "custom"
plot_series(df_exp1[mask], "order", "cache", directory_output, "exp_until_cache", logx=True)

